# V6 formatter fine-tuning and qualification (Colab)

Use a GPU runtime. This notebook trains the balanced contract-v6 controls plus the paired real-derived set against the V5 base, then evaluates V5 and V6 on independent challenge, hard-eval, mixed held-out, and real-derived held-out sets. The packaged safety protocol includes Unicode-aware source-copy and deletion checks.

Before starting, create `vaani-v6/v6-colab-balanced-contract-v6-gold-gate-v1-20261003/` in Drive. Copy `archflow-v6-training-code.zip`, `v6-formatter-training-data.zip`, `bundle_manifest.json`, and `v5-formatter-smollm2-360m-minimal.zip` there. The data archive contains text-only contract-v6 balanced splits, real-derived splits, the v3 spelling challenge, hard-eval, and validated foundation/hard replay; it contains no audio or model weights. The notebook verifies all bundle hashes before extraction, validates foundation rows, excludes held-out sources, resumes a saved checkpoint after Colab disconnects, and saves checkpoints/reports to Drive.

In [ ]:
%pip -q install transformers==5.17.0 peft==0.20.0 accelerate==1.15.0 safetensors
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before training')
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile, sys, json, tempfile
ROOT = Path('/content/drive/MyDrive/vaani-v6/v6-colab-balanced-contract-v6-gold-gate-v1-20261003')
CODE_ZIP = ROOT / 'archflow-v6-training-code.zip'
DATA_ZIP = ROOT / 'v6-formatter-training-data.zip'
MANIFEST = ROOT / 'bundle_manifest.json'
STAGING = Path(tempfile.mkdtemp(prefix='vaani-v6-verified-inputs-'))
CODE = STAGING / 'ArchFlow'
DATA = STAGING / 'data'
SYNTH = DATA / 'synthetic'
REAL = DATA / 'real'
MODEL = STAGING / 'v5-formatter-smollm2-360m'
MODEL_ZIP = ROOT / 'v5-formatter-smollm2-360m-minimal.zip'
EXPECTED_SHA256 = {
    CODE_ZIP.name: '33108bef875140bb31fe402b0b4ace70c44a330143a773f9b0a08903b7485015',
    MODEL_ZIP.name: '114c574fc9f0bbddb28688215aed02445f4b1e330831a65a14a5e7a229d95361',
    DATA_ZIP.name: 'd37b67d4191ead4f4544d964733d86ee0c51d37e0c3273525e05a9e81789712a',
}
import hashlib
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): digest.update(block)
    return digest.hexdigest()
if not MANIFEST.is_file(): raise FileNotFoundError(MANIFEST)
if file_sha256(MANIFEST) != 'cc4d721faef318c351e2493b373ee88b18e99ff5476208d0460585b3a03f935e': raise ValueError('Bundle manifest hash mismatch')
bundle_manifest = json.loads(MANIFEST.read_text())
for bundle in (CODE_ZIP, MODEL_ZIP, DATA_ZIP):
    if not bundle.is_file(): raise FileNotFoundError(bundle)
    actual = file_sha256(bundle)
    if actual != EXPECTED_SHA256[bundle.name]: raise ValueError(f'Bundle hash mismatch: {bundle.name}')
for key, path in [('code_bundle', CODE_ZIP), ('data_bundle', DATA_ZIP), ('model_bundle', MODEL_ZIP)]:
    entry = bundle_manifest[key]
    if entry['file'] != path.name or entry['sha256'] != EXPECTED_SHA256[path.name]: raise ValueError(f'Manifest mismatch: {path.name}')
with zipfile.ZipFile(DATA_ZIP) as z: z.extractall(STAGING)
with zipfile.ZipFile(MODEL_ZIP) as z: z.extractall(STAGING)
REPLAY = DATA / 'hard-replay.jsonl'
FOUNDATION = DATA / 'foundation-seed-v1.jsonl'
CHALLENGE = DATA / 'challenge-contract-v3.jsonl'
HARD_EVAL = DATA / 'hard-eval.jsonl'
OUT = ROOT / 'candidate-v6-balanced-contract-v6'
required = [CODE_ZIP, DATA_ZIP, MODEL / 'config.json', REPLAY, CHALLENGE, HARD_EVAL] + [SYNTH / f'{s}.jsonl' for s in ('train','dev','test')] + [REAL / f'{s}.jsonl' for s in ('train','dev','test')]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Copy the listed code/model/data files into Drive first: ' + ', '.join(missing))
if OUT.exists():
    if any(OUT.glob('v5-*.json')) or any(OUT.glob('v6-*.json')) or any(OUT.glob('comparison-*.json')): raise FileExistsError(f'Evaluation already exists; refusing to overwrite: {OUT}')
    checkpoints = list((OUT/'adapter').glob('checkpoint-*'))
    unexpected = [path for path in OUT.iterdir() if path.name != 'run-environment.json']
    if not checkpoints and unexpected: raise FileExistsError(f'Run directory exists without a resumable checkpoint: {OUT}')
CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z: z.extractall(CODE)
sys.path.insert(0, str(CODE / 'training/cleanup-llm/scripts'))
print('Inputs found; code extracted; output:', OUT)

In [ ]:
import subprocess, json
import importlib.metadata, platform
def run(args):
    print('+', ' '.join(map(str, args)))
    subprocess.run(list(map(str, args)), check=True, cwd=CODE)
validator = CODE / 'tools/validate_v6_foundation.py'
run([sys.executable, validator, FOUNDATION, '--require-approved'])
trainer = CODE / 'tools/train_v6_seq2seq.py'
OUT.mkdir(parents=True, exist_ok=True)
environment = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
    'packages': {name: importlib.metadata.version(name) for name in ('transformers', 'peft', 'accelerate', 'safetensors')},
}
checkpoints = sorted((OUT/'adapter').glob('checkpoint-*'), key=lambda p: int(p.name.rsplit('-', 1)[-1])) if (OUT/'adapter').exists() else []
environment_path = OUT / 'run-environment.json'
if environment_path.exists():
    if json.loads(environment_path.read_text()) != environment: raise RuntimeError('Runtime differs from the recorded checkpoint environment; refusing to resume')
else:
    if checkpoints: raise RuntimeError('Checkpoint has no recorded runtime fingerprint; refusing to resume without provenance')
    environment_path.write_text(json.dumps(environment, indent=2, sort_keys=True) + '\n')
resume_args = ['--resume-from-checkpoint', checkpoints[-1]] if checkpoints else []
subprocess.run([sys.executable, str(trainer), '--train', str(SYNTH/'train.jsonl'), str(REAL/'train.jsonl'), '--dev', str(SYNTH/'dev.jsonl'), str(REAL/'dev.jsonl'), '--replay', str(REPLAY), '--replay', str(FOUNDATION), '--replay-factor', '3', '--exclude', str(CHALLENGE), str(HARD_EVAL), '--model', str(MODEL), '--out', str(OUT/'adapter'), '--steps', '3000', '--eval-steps', '500', *resume_args], check=True, cwd=CODE)

In [ ]:
evaluator = CODE / 'tools/eval_v6_seq2seq.py'
comparator = CODE / 'tools/compare_v6_seq2seq.py'
def paired(suite, data, fallback=None):
    v5, v6 = OUT/f'v5-{suite}.json', OUT/f'v6-{suite}.json'
    run([sys.executable,evaluator,'--model',MODEL,'--protocol','v5','--data',*data,'--report',v5])
    run([sys.executable,evaluator,'--model',MODEL,'--adapter',OUT/'adapter','--protocol','v6','--data',*data,'--report',v6])
    cmd=[sys.executable,comparator,'--v5',v5,'--v6',v6,'--suite',suite,'--report',OUT/f'comparison-{suite}.json']
    if fallback is not None: cmd += ['--max-copy-fallback-rate',str(fallback)]
    # Keep collecting aggregate evidence even if one suite rejects the candidate.
    result=subprocess.run(list(map(str,cmd)),cwd=CODE,check=False)
    return result.returncode
OUT.mkdir(parents=True, exist_ok=True)
suite_results = {}
for name, data, limit in [
    ('challenge', [CHALLENGE], None),
    ('hard-eval', [HARD_EVAL], None),
    ('heldout-test', [SYNTH/'test.jsonl', REAL/'test.jsonl'], 0.05),
    ('real-test', [REAL/'test.jsonl'], 0.01),
]:
    suite_results[name] = paired(name, data, limit)
print('comparison exit codes (0 means suite gate passed):', suite_results)
print('Aggregate reports:', *sorted(str(p.name) for p in OUT.glob('comparison-*.json')))
if any(code != 0 for code in suite_results.values()): raise RuntimeError('V6 formatter candidate failed one or more promotion gates; keep it quarantined')

## Reading the result

A successful training run is not a promotion. Check every `comparison-*.json`; V6 must improve over paired V5, preserve protected content, emit no content absent from the source, omit no content required by the gold target, and meet fallback limits on the mixed and real-derived held-out suites. Keep rejected adapters for experiment records and do not install them. The notebook stores model checkpoints in Drive, so confirm adequate quota before starting.